In [14]:
from dotenv import load_dotenv
load_dotenv()

True

In [15]:
import pandas as pd

# QA based on Agentic AI text
inputs = [
    "What is Agentic AI?",
    "What is the main difference between Agentic AI and traditional AI systems?",
    "What are the key characteristics of Agentic AI?",
    "What percentage of human intervention is required in Agentic AI systems?",
    "What are some common applications of Agentic AI?",
    "Which technologies are commonly used to build Agentic AI systems?",
    "What role does memory play in Agentic AI?",
    "How does an Agentic AI system typically process a user request?",
]

outputs = [
    "Agentic AI is an advanced form of artificial intelligence that can independently make decisions, plan tasks, and take actions to achieve specific goals with minimal human intervention.",
    
    "Unlike traditional AI systems that only respond to direct instructions, Agentic AI systems can analyze situations, reason through problems, adapt to changing environments, and execute multi-step workflows autonomously.",
    
    "Key characteristics of Agentic AI include autonomy, goal-oriented behavior, reasoning and decision-making, memory and context awareness, and tool integration.",
    
    "Agentic AI systems require minimal human intervention because they can operate autonomously and independently handle tasks and decisions.",
    
    "Applications of Agentic AI include customer support automation, personal AI assistants, autonomous research systems, healthcare support systems, financial analysis and trading, smart robotics and automation, and software development assistants.",
    
    "Technologies commonly used in building Agentic AI systems include Large Language Models (LLMs), Retrieval-Augmented Generation (RAG), Vector Databases, LangChain, FastAPI, ChromaDB, and OpenAI, Grok, or Gemini models.",
    
    "Memory in Agentic AI allows agents to remember previous interactions and use historical context for better responses and decision-making.",
    
    "An Agentic AI system processes a request by analyzing the question, retrieving relevant data, planning actions, using external tools or APIs if needed, and generating the final response."
]

# Create dataset
qa_pairs = [{"question": q, "answer": a} for q, a in zip(inputs, outputs)]

df = pd.DataFrame(qa_pairs)

# Save CSV
csv_path = "C:/code/LLMops_RAG/data/goldens.csv"
df.to_csv(csv_path, index=False)

print(df)
print(f"\nCSV saved at: {csv_path}")

                                            question  \
0                                What is Agentic AI?   
1  What is the main difference between Agentic AI...   
2    What are the key characteristics of Agentic AI?   
3  What percentage of human intervention is requi...   
4   What are some common applications of Agentic AI?   
5  Which technologies are commonly used to build ...   
6          What role does memory play in Agentic AI?   
7  How does an Agentic AI system typically proces...   

                                              answer  
0  Agentic AI is an advanced form of artificial i...  
1  Unlike traditional AI systems that only respon...  
2  Key characteristics of Agentic AI include auto...  
3  Agentic AI systems require minimal human inter...  
4  Applications of Agentic AI include customer su...  
5  Technologies commonly used in building Agentic...  
6  Memory in Agentic AI allows agents to remember...  
7  An Agentic AI system processes a request by an...  


In [17]:
from langsmith import Client

client = Client()
dataset_name = "AgenticAIReportGoldens1"

# Store
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Input and expected output pairs for AgenticAIReport",
)
client.create_examples(
    inputs=[{"question": q} for q in inputs],
    outputs=[{"answer": a} for a in outputs],
    dataset_id=dataset.id,
)

{'example_ids': ['9287d0d7-c83c-42dc-a750-7c7f5d6cc9be',
  'e0a536f0-f9f3-48f6-83fd-79c9aa82581d',
  'cf6c0508-bd2e-49c1-8d32-7a05860630cc',
  '1c8766a4-40a7-4cf5-9272-0481d776061c',
  '87110a1a-a36d-4ed9-b277-f4fdd7f6f520',
  'b6c66c0a-d363-4dc0-a920-10856f657cc4',
  'fcac5cb0-d892-4d08-bd3e-2f1b62691cfa',
  '529eb124-5ccc-4ecd-a125-baedf9fbb0cd'],
 'count': 8,
 'as_of': '2026-05-24T16:46:54.289895746Z'}

In [18]:
import sys
sys.path.append("C:/code/LLMops_RAG")

from pathlib import Path
from multi_doc_chat.src.document_ingestion.data_ingestion import ChatIngestor
from multi_doc_chat.src.document_chat.retrieval import ConversationalRAG
import os

# Simple file adapter for local file paths
class LocalFileAdapter:
    """Adapter for local file paths to work with ChatIngestor."""
    def __init__(self, file_path: str):
        self.path = Path(file_path)
        self.name = self.path.name
    
    def getbuffer(self) -> bytes:
        return self.path.read_bytes()


def answer_ai_report_question(
    inputs: dict,
    data_path: str = "C:/code/LLMops_RAG/data/Agentic AI.txt",
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
    k: int = 5
) -> dict:
    """
    Answer questions about the AI Engineering Report using RAG.
    
    Args:
        inputs: Dictionary containing the question, e.g., {"question": "What is RAG?"}
        data_path: Path to the AI Engineering Report text file
        chunk_size: Size of text chunks for splitting
        chunk_overlap: Overlap between chunks
        k: Number of documents to retrieve
    
    Returns:
        Dictionary with the answer, e.g., {"answer": "RAG stands for..."}
    """
    try:
        # Extract question from inputs
        question = inputs.get("question", "")
        if not question:
            return {"answer": "No question provided"}
        
        # Check if file exists
        if not Path(data_path).exists():
            return {"answer": f"Data file not found: {data_path}"}
        
        # Create file adapter
        file_adapter = LocalFileAdapter(data_path)
        
        # Build index using ChatIngestor
        ingestor = ChatIngestor(
            temp_base="data",
            faiss_base="faiss_index",
            use_session_dirs=True
        )
        
        # Build retriever
        ingestor.built_retriver(
            uploaded_files=[file_adapter],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            k=k
        )
        
        # Get session ID and index path
        session_id = ingestor.session_id
        index_path = f"faiss_index/{session_id}"
        
        # Create RAG instance and load retriever
        rag = ConversationalRAG(session_id=session_id)
        rag.load_retriever_from_faiss(
            index_path=index_path,
            k=k,
            index_name=os.getenv("FAISS_INDEX_NAME", "index")
        )
        
        # Get answer
        answer = rag.invoke(question, chat_history=[])
        
        return {"answer": answer}
        
    except Exception as e:
        return {"answer": f"Error: {str(e)}"}

In [19]:
# Test the function with a sample question
test_input = {"question": "What are the key characteristics of Agentic AI?"}
result = answer_ai_report_question(test_input)
print("Question:", test_input["question"])
print("\nAnswer:", result["answer"])

{"timestamp": "2026-05-24T16:55:38.311933Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-05-24T16:55:38.316847Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_Hk..."}, "timestamp": "2026-05-24T16:55:38.316847Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-05-24T16:55:38.329525Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260524_222538_706cd651", "temp_dir": "data\\session_20260524_222538_706cd651", "faiss_dir": "faiss_index\\session_20260524_222538_706cd651", "sessionized": true, "timestamp": "2026-05-24T16:55:38.339040Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "Agentic AI.txt", "saved_as": "data\\session_20260524_222538_706cd651\\1c67cba8.txt", "timestamp": "2026-05-24T16:55:38.342222Z", "level": "info", "event": "File saved for ingestion"}
{"count": 1,

Question: What are the key characteristics of Agentic AI?

Answer: The key characteristics of Agentic AI include Autonomy, Goal-Oriented Behavior, Reasoning and Decision-Making, Memory and Context Awareness, and Tool Integration. These characteristics enable Agentic AI systems to operate independently, make decisions, and achieve specific goals with minimal human intervention.


In [20]:
from langsmith.evaluation import evaluate

In [22]:
print("Testing all questions from the dataset:\n")
for i, q in enumerate(inputs, 1):
    test_input = {"question": q}
    result = answer_ai_report_question(test_input)
    print(f"Q{i}: {q}")
    print(f"A{i}: {result['answer']}\n")
    print("-" * 80 + "\n")

{"timestamp": "2026-05-24T17:00:36.227785Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-05-24T17:00:36.229810Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_Hk..."}, "timestamp": "2026-05-24T17:00:36.232842Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-05-24T17:00:36.242360Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260524_223036_16ef146b", "temp_dir": "data\\session_20260524_223036_16ef146b", "faiss_dir": "faiss_index\\session_20260524_223036_16ef146b", "sessionized": true, "timestamp": "2026-05-24T17:00:36.245910Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "Agentic AI.txt", "saved_as": "data\\session_20260524_223036_16ef146b\\f591024c.txt", "timestamp": "2026-05-24T17:00:36.250323Z", "level": "info", "event": "File saved for ingestion"}
{"count": 1,

Testing all questions from the dataset:



HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-tra

Q1: What is Agentic AI?
A1: Agentic AI is an advanced form of artificial intelligence that can independently make decisions, plan tasks, and take actions to achieve specific goals with minimal human intervention. It can analyze situations, reason through problems, adapt to changing environments, and execute multi-step workflows autonomously. Agentic AI combines capabilities such as memory, reasoning, planning, tool usage, and continuous learning.

--------------------------------------------------------------------------------



HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-tra

...

Q6: Which technologies are commonly used to build Agentic AI systems?
A6: Technologies commonly used to build Agentic AI systems include Large Language Models (LLMs), Retrieval-Augmented Generation (RAG), Vector Databases, LangChain, FastAPI, ChromaDB, and OpenAI, Grok, or Gemini models. These technologies enable Agentic AI systems to operate independently and achieve specific goals with minimal human intervention.

--------------------------------------------------------------------------------



HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-tra

Q7: What role does memory play in Agentic AI?
A7: Memory plays a role in Agentic AI by enabling the AI agent to remember previous interactions and use historical context for better responses and decisions. This is referred to as "Memory and Context Awareness" in Agentic AI systems. It allows the AI to learn from past experiences and adapt to changing environments.

--------------------------------------------------------------------------------



HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-tra

Q8: How does an Agentic AI system typically process a user request?
A8: An Agentic AI system typically processes a user request by analyzing the request, retrieving relevant data, planning the required actions, using external tools or APIs if needed, and generating a final response. The system works toward achieving predefined objectives by planning and executing multiple steps. The agent can also remember previous interactions and use historical context for better responses and decisions.

--------------------------------------------------------------------------------



In [31]:
from langsmith.evaluation import evaluate
from langchain.smith import RunEvalConfig

dataset_name = "AgenticAIReportGoldens1"

eval_config = RunEvalConfig(
    evaluators=[
        "qa",
        "cot_qa",
    ]
)

experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=eval_config.evaluators,
    experiment_prefix="test-agenticAIReport-qa-rag",
    metadata={
        "variant": "RAG with FAISS and AI Engineering Report",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

View the evaluation results for experiment: 'test-agenticAIReport-qa-rag-9686ae23' at:
https://smith.langchain.com/o/8576ae9f-5af0-4049-b58e-76a20e9af3a2/datasets/0621d246-cdcc-4c1a-9562-5b2557f5d866/compare?selectedSessions=aa282d84-ceb2-48d5-8cba-9c705d805574




TypeError: 'qa' is not a callable object